# Level 2: tiny models from scratch, bits per character (BPC)

**Settings:** Accelerator **GPU T4 x2**, Internet **off** is fine (no downloads), Persistence: *Files only* (so `/kaggle/working/runs` survives restarts).

**Inputs:** `bigo-code` (repo zip) and `bigo-level2` (output of `python -m eval.level2.prepare`, one folder per tokenizer).

Each run trains a ~42M-param GPT (d=512, 8 layers, ctx 512) on the same training characters, then writes BPC rows to `/kaggle/working/results/level2_bpc.csv`. Finished runs are skipped, so re-running after a restart resumes from checkpoints.

fp16 + GradScaler, SDPA attention, DDP over both T4s. **Stop the session when the last cell finishes.**

In [ ]:
import glob, os, json, subprocess
CODE = glob.glob('/kaggle/input/bigo-code*')[0]
DATA = glob.glob('/kaggle/input/bigo-level2*')[0]
if not os.path.exists('/kaggle/working/repo'):
    subprocess.run(['cp', '-r', CODE, '/kaggle/working/repo'], check=True)
os.chdir('/kaggle/working/repo')
TOKS = sorted(d for d in os.listdir(DATA) if os.path.exists(f'{DATA}/{d}/meta.json'))
for t in TOKS:
    m = json.load(open(f'{DATA}/{t}/meta.json'))
    print(t, 'V', m['vocab_size'], 'train tokens', f"{m['train_tokens']:,}", 'chars', f"{m['train_chars']:,}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

Run plan. `EPOCHS` = passes over the shared training characters (same for every tokenizer). First do a 1-run timing check (`QUICK=True`), then set `QUICK=False`.

In [ ]:
QUICK = True
EPOCHS = 3
SEEDS = [0, 1]
ORDER = [t for t in ['plain_32k', 'l1_32k', 'l12_32k', 'l123_32k', 'l1234_32k', 'bytes'] if t in TOKS] + \n        [t for t in TOKS if t not in ['plain_32k', 'l1_32k', 'l12_32k', 'l123_32k', 'l1234_32k', 'bytes']]
print(ORDER)

In [ ]:
def run(tok, seed, extra=()):
    out = f'/kaggle/working/runs/{tok}_s{seed}'
    if os.path.exists(f'{out}/bpc.json'):
        print('skip', tok, seed); return
    cmd = ['torchrun', '--nproc_per_node', '2', '-m', 'eval.level2.train', '--data', f'{DATA}/{tok}', '--out', out,
           '--seed', str(seed), '--epochs', str(EPOCHS), '--results', '/kaggle/working/results', *extra]
    print(' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)

if QUICK:
    run(ORDER[0], 0, ['--max-steps', '100', '--out', '/kaggle/working/_timing/run', '--results', '/kaggle/working/_timing'])  # prints tok/s; kept out of results/
else:
    for seed in SEEDS:
        for tok in ORDER:
            run(tok, seed)

In [ ]:
!tail -n +1 /kaggle/working/results/level2_bpc.csv | cut -d, -f1-6
!cd /kaggle/working && zip -qr level2_out.zip results runs/*/log.csv runs/*/bpc.json && ls -la level2_out.zip